LAB 5:
NHẬN DẠNG QUẦN ÁO GIÀY DÉP THỜI TRANG VỚI BỘ DŨ LIỆU
FASHION-MNIST, DÙNG THƯ VIỆN PYTORCH HUẤN LUYỆN MÔ
HÌNH TRÊN GPU CỦA FIT-LAB

In [9]:
# Cài đặt các thư viện cần thiết
!pip install --upgrade torch torchvision matplotlib matplotlib-inline numpy

# Import các thư viện
import torch
from torchvision import datasets
import matplotlib
matplotlib.use("Agg")  # tránh lỗi backend GUI
import matplotlib.pyplot as plt
import numpy as np

# Tải dữ liệu FashionMNIST
data_folder = '~/data/FMNIST'
fmnist = datasets.FashionMNIST(data_folder, download=True, train=True)

# Lấy ảnh và nhãn
tr_images = fmnist.data
tr_targets = fmnist.targets

# Kiểm tra thông tin dữ liệu
unique_values = tr_targets.unique()
print(f'tr_images & tr_targets:\n\tX - {tr_images.shape}\n\tY - {tr_targets.shape}\n\tY - Unique Values : {unique_values}')
print(f'TASK:\n\t{len(unique_values)} class Classification')
print(f'UNIQUE CLASSES:\n\t{fmnist.classes}')

# Trực quan hóa dữ liệu: tạo lưới 10x10 và lưu ra file
R, C = len(tr_targets.unique()), 10
fig, axes = plt.subplots(R, C, figsize=(12,12))

for label_class in range(R):
    label_x_rows = np.where(tr_targets == label_class)[0]
    for c in range(C):
        ix = np.random.choice(label_x_rows)
        x = tr_images[ix]
        axes[label_class, c].imshow(x, cmap='gray')
        axes[label_class, c].axis('off')
        if c == 0:  # ghi tên lớp ở cột đầu tiên
            axes[label_class, c].set_ylabel(fmnist.classes[label_class], fontsize=8)

plt.tight_layout()
plt.savefig("fashionmnist_grid.png")  # lưu ra file ảnh
print("✅ Đã lưu hình ảnh trực quan hóa vào fashionmnist_grid.png")


  Using cached numpy-2.0.2-cp39-cp39-win_amd64.whl.metadata (59 kB)
   ---------------------------------------- 0.0/7.8 MB ? eta -:--:--
   -- ------------------------------------- 0.5/7.8 MB 8.5 MB/s eta 0:00:01
   -------- ------------------------------- 1.6/7.8 MB 4.7 MB/s eta 0:00:02
   ------------- -------------------------- 2.6/7.8 MB 4.7 MB/s eta 0:00:02
   ---------------- ----------------------- 3.1/7.8 MB 4.4 MB/s eta 0:00:02
   -------------------- ------------------- 3.9/7.8 MB 4.2 MB/s eta 0:00:01
   ------------------------ --------------- 4.7/7.8 MB 4.1 MB/s eta 0:00:01
   ----------------------------- ---------- 5.8/7.8 MB 4.2 MB/s eta 0:00:01
   ---------------------------------- ----- 6.8/7.8 MB 4.2 MB/s eta 0:00:01
   -------------------------------------- - 7.6/7.8 MB 4.2 MB/s eta 0:00:01
   ---------------------------------------- 7.8/7.8 MB 3.9 MB/s eta 0:00:00
Using cached numpy-2.0.2-cp39-cp39-win_amd64.whl (15.9 MB)
  Attempting uninstall: numpy
    Found exis

  You can safely remove it manually.
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
rasa 3.6.21 requires matplotlib<3.6,>=3.1, but you have matplotlib 3.9.4 which is incompatible.
rasa 3.6.21 requires numpy<1.25.0,>=1.19.2; python_version >= "3.8" and python_version < "3.11", but you have numpy 2.0.2 which is incompatible.
rasa 3.6.21 requires packaging<21.0,>=20.0, but you have packaging 26.2 which is incompatible.
rasa 3.6.21 requires prompt-toolkit<3.0.29,>=3.0, but you have prompt-toolkit 3.0.52 which is incompatible.
scipy 1.10.1 requires numpy<1.27.0,>=1.19.5, but you have numpy 2.0.2 which is incompatible.
tensorflow-intel 2.12.0 requires numpy<1.24,>=1.22, but you have numpy 2.0.2 which is incompatible.

[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install -

✅ Đã lưu hình ảnh trực quan hóa vào fashionmnist_grid.png


3. Huấn luyện mạng nơ-ron
Để huấn luyện mạng nơ-ron, chúng ta phải thực hiện các bước sau:
1. Import các gói có liên quan.
2. Xây dựng tập dữ liệu có thể tìm nạp dữ liệu từng ảnh một lần.
3. Gói DataLoader từ tập dữ liệu.
4. Xây dựng mô hình, sau đó xác định hàm sai số và trình tối ưu hóa.
5. Xác định hai hàm để huấn luyện và xác thực một bó dữ liệu tương ứng.
6. Xác định hàm sẽ tính toán độ chính xác của dữ liệu.
7. Thực hiện cập nhật trọng số theo từng bó dữ liệu tăng dần qua từng epoch.

In [10]:
# Cài đặt thư viện cần thiết
!pip install --upgrade torch torchvision matplotlib matplotlib-inline numpy

# Import
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import SGD
import matplotlib
matplotlib.use("Agg")  # tránh lỗi backend GUI
import matplotlib.pyplot as plt
import numpy as np
from torchvision import datasets

# Thiết bị
device = "cuda" if torch.cuda.is_available() else "cpu"

# 1. Tải dữ liệu FashionMNIST
data_folder = '~/data/FMNIST'
fmnist = datasets.FashionMNIST(data_folder, download=True, train=True)
tr_images, tr_targets = fmnist.data, fmnist.targets

print("Shape ảnh:", tr_images.shape)
print("Shape nhãn:", tr_targets.shape)
print("Các lớp:", fmnist.classes)

# 2. Xây dựng Dataset tùy chỉnh
class FMNISTDataset(Dataset):
    def __init__(self, x, y):
        x = x.float() / 255.0  # chuẩn hóa về [0,1]
        x = x.view(-1, 28*28)  # flatten
        self.x, self.y = x, y
    def __getitem__(self, ix):
        return self.x[ix].to(device), self.y[ix].to(device)
    def __len__(self):
        return len(self.x)

# 3. DataLoader
def get_data():
    train = FMNISTDataset(tr_images, tr_targets)
    trn_dl = DataLoader(train, batch_size=32, shuffle=True)
    return trn_dl

# 4. Định nghĩa mô hình, loss, optimizer
def get_model():
    model = nn.Sequential(
        nn.Linear(28*28, 1000),
        nn.ReLU(),
        nn.Linear(1000, 10)
    ).to(device)
    loss_fn = nn.CrossEntropyLoss()
    optimizer = SGD(model.parameters(), lr=1e-2)
    return model, loss_fn, optimizer

# 5. Hàm train batch
def train_batch(x, y, model, opt, loss_fn):
    model.train()
    prediction = model(x)
    batch_loss = loss_fn(prediction, y)
    batch_loss.backward()
    opt.step()
    opt.zero_grad()
    return batch_loss.item()

# 6. Hàm accuracy
@torch.no_grad()
def accuracy(x, y, model):
    model.eval()
    prediction = model(x)
    _, argmaxes = prediction.max(-1)
    return (argmaxes == y).cpu().numpy().tolist()

# 7. Huấn luyện mô hình
trn_dl = get_data()
model, loss_fn, optimizer = get_model()
losses, accuracies = [], []

for epoch in range(5):
    print("Epoch:", epoch+1)
    epoch_losses, epoch_accuracies = [], []
    
    # Train
    for x, y in trn_dl:
        batch_loss = train_batch(x, y, model, optimizer, loss_fn)
        epoch_losses.append(batch_loss)
    epoch_loss = np.mean(epoch_losses)
    
    # Eval
    for x, y in trn_dl:
        is_correct = accuracy(x, y, model)
        epoch_accuracies.extend(is_correct)
    epoch_accuracy = np.mean(epoch_accuracies)
    
    losses.append(epoch_loss)
    accuracies.append(epoch_accuracy)

# 8. Vẽ kết quả và lưu ra file
epochs = np.arange(5)+1
plt.figure(figsize=(20,5))

plt.subplot(121)
plt.title('Loss value over increasing epochs')
plt.plot(epochs, losses, label='Training Loss')
plt.legend()

plt.subplot(122)
plt.title('Accuracy value over increasing epochs')
plt.plot(epochs, accuracies, label='Training Accuracy')
plt.gca().set_yticklabels(['{:.0f}%'.format(x*100) for x in plt.gca().get_yticks()])
plt.legend()

plt.tight_layout()
plt.savefig("training_results.png")
print("✅ Đã lưu biểu đồ huấn luyện vào training_results.png")



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Shape ảnh: torch.Size([60000, 28, 28])
Shape nhãn: torch.Size([60000])
Các lớp: ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat', 'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']
Epoch: 1
Epoch: 2
Epoch: 3
Epoch: 4
Epoch: 5


C:\Users\Admin\AppData\Local\Temp\ipykernel_22520\1681282268.py:109: UserWarning: FixedFormatter should only be used together with FixedLocator
  plt.gca().set_yticklabels(['{:.0f}%'.format(x*100) for x in plt.gca().get_yticks()])


✅ Đã lưu biểu đồ huấn luyện vào training_results.png


In [12]:
# Cài đặt thư viện cần thiết
!pip install --upgrade torch torchvision matplotlib matplotlib-inline numpy

# Import
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import SGD
import matplotlib
matplotlib.use("Agg")  # tránh lỗi backend GUI
import matplotlib.pyplot as plt
import numpy as np
from torchvision import datasets

# Thiết bị
device = "cuda" if torch.cuda.is_available() else "cpu"

# 1. Tải dữ liệu FashionMNIST
data_folder = '~/data/FMNIST'
fmnist = datasets.FashionMNIST(data_folder, download=True, train=True)
tr_images, tr_targets = fmnist.data, fmnist.targets

print("Shape ảnh:", tr_images.shape)
print("Shape nhãn:", tr_targets.shape)
print("Các lớp:", fmnist.classes)

# 2. Dataset tùy chỉnh (chuẩn hóa ảnh về [0,1])
class FMNISTDataset(Dataset):
    def __init__(self, x, y):
        x = x.float()/255.0
        x = x.view(-1, 28*28)
        self.x, self.y = x, y
    def __getitem__(self, ix):
        return self.x[ix].to(device), self.y[ix].to(device)
    def __len__(self):
        return len(self.x)

# 3. DataLoader
def get_data():
    train = FMNISTDataset(tr_images, tr_targets)
    trn_dl = DataLoader(train, batch_size=32, shuffle=True)
    return trn_dl

# 4. Mô hình, loss, optimizer
def get_model():
    model = nn.Sequential(
        nn.Linear(28*28, 1000),
        nn.ReLU(),
        nn.Linear(1000, 10)
    ).to(device)
    loss_fn = nn.CrossEntropyLoss()
    optimizer = SGD(model.parameters(), lr=1e-2)
    return model, loss_fn, optimizer

# 5. Train batch
def train_batch(x, y, model, opt, loss_fn):
    model.train()
    prediction = model(x)
    batch_loss = loss_fn(prediction, y)
    batch_loss.backward()
    opt.step()
    opt.zero_grad()
    return batch_loss.item()

# 6. Accuracy
@torch.no_grad()
def accuracy(x, y, model):
    model.eval()
    prediction = model(x)
    _, argmaxes = prediction.max(-1)
    return (argmaxes == y).cpu().numpy().tolist()

# 7. Huấn luyện
trn_dl = get_data()
model, loss_fn, optimizer = get_model()
losses, accuracies = [], []

for epoch in range(5):
    print("Epoch:", epoch+1)
    epoch_losses, epoch_accuracies = [], []
    
    # Train
    for x, y in trn_dl:
        batch_loss = train_batch(x, y, model, optimizer, loss_fn)
        epoch_losses.append(batch_loss)
    epoch_loss = np.mean(epoch_losses)
    
    # Eval
    for x, y in trn_dl:
        is_correct = accuracy(x, y, model)
        epoch_accuracies.extend(is_correct)
    epoch_accuracy = np.mean(epoch_accuracies)
    
    losses.append(epoch_loss)
    accuracies.append(epoch_accuracy)

# 8. Vẽ kết quả và lưu file
epochs = np.arange(5)+1
plt.figure(figsize=(20,5))

plt.subplot(121)
plt.title('Loss value over increasing epochs')
plt.plot(epochs, losses, label='Training Loss')
plt.legend()

plt.subplot(122)
plt.title('Accuracy value over increasing epochs')
plt.plot(epochs, accuracies, label='Training Accuracy')
plt.gca().set_yticklabels(['{:.0f}%'.format(x*100) for x in plt.gca().get_yticks()])
plt.legend()

plt.tight_layout()
plt.savefig("training_results.png")
print("✅ Đã lưu biểu đồ huấn luyện vào training_results.png")



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Shape ảnh: torch.Size([60000, 28, 28])
Shape nhãn: torch.Size([60000])
Các lớp: ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat', 'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']
Epoch: 1
Epoch: 2
Epoch: 3
Epoch: 4
Epoch: 5
✅ Đã lưu biểu đồ huấn luyện vào training_results.png


C:\Users\Admin\AppData\Local\Temp\ipykernel_22520\383945735.py:109: UserWarning: FixedFormatter should only be used together with FixedLocator
  plt.gca().set_yticklabels(['{:.0f}%'.format(x*100) for x in plt.gca().get_yticks()])


In [13]:
# Cài đặt thư viện cần thiết
!pip install --upgrade torch torchvision matplotlib matplotlib-inline numpy

# Import
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
import matplotlib
matplotlib.use("Agg")  # tránh lỗi backend GUI
import matplotlib.pyplot as plt
import numpy as np
from torchvision import datasets

# Thiết bị
device = "cuda" if torch.cuda.is_available() else "cpu"

# 1. Tải dữ liệu FashionMNIST (train + validation)
data_folder = '~/data/FMNIST'
fmnist = datasets.FashionMNIST(data_folder, download=True, train=True)
val_fmnist = datasets.FashionMNIST(data_folder, download=True, train=False)

tr_images, tr_targets = fmnist.data, fmnist.targets
val_images, val_targets = val_fmnist.data, val_fmnist.targets

print("Train:", tr_images.shape, tr_targets.shape)
print("Validation:", val_images.shape, val_targets.shape)
print("Classes:", fmnist.classes)

# 2. Dataset tùy chỉnh
class FMNISTDataset(Dataset):
    def __init__(self, x, y):
        x = x.float()/255.0
        x = x.view(-1, 28*28)
        self.x, self.y = x, y
    def __getitem__(self, ix):
        return self.x[ix].to(device), self.y[ix].to(device)
    def __len__(self):
        return len(self.x)

# 3. DataLoader
def get_data():
    train = FMNISTDataset(tr_images, tr_targets)
    trn_dl = DataLoader(train, batch_size=32, shuffle=True)
    val = FMNISTDataset(val_images, val_targets)
    val_dl = DataLoader(val, batch_size=len(val_images), shuffle=False)
    return trn_dl, val_dl

# 4. Mô hình, loss, optimizer
def get_model():
    model = nn.Sequential(
        nn.Linear(28*28, 1000),
        nn.ReLU(),
        nn.Linear(1000, 10)
    ).to(device)
    loss_fn = nn.CrossEntropyLoss()
    optimizer = Adam(model.parameters(), lr=1e-2)
    return model, loss_fn, optimizer

# 5. Train batch
def train_batch(x, y, model, opt, loss_fn):
    model.train()
    prediction = model(x)
    batch_loss = loss_fn(prediction, y)
    batch_loss.backward()
    opt.step()
    opt.zero_grad()
    return batch_loss.item()

# 6. Accuracy
@torch.no_grad()
def accuracy(x, y, model):
    model.eval()
    prediction = model(x)
    _, argmaxes = prediction.max(-1)
    return (argmaxes == y).cpu().numpy().tolist()

# 7. Validation loss
@torch.no_grad()
def val_loss(X, y, model, loss_fn):
    prediction = model(X)
    return loss_fn(prediction, y).item()

# 8. Huấn luyện + đánh giá
trn_dl, val_dl = get_data()
model, loss_fn, optimizer = get_model()

train_losses, train_accuracies = [], []
val_losses, val_accuracies = [], []

for epoch in range(5):
    print("Epoch:", epoch+1)
    train_epoch_losses, train_epoch_accuracies = [], []
    val_epoch_losses, val_epoch_accuracies = [], []
    
    # Train
    for x, y in trn_dl:
        batch_loss = train_batch(x, y, model, optimizer, loss_fn)
        train_epoch_losses.append(batch_loss)
        is_correct = accuracy(x, y, model)
        train_epoch_accuracies.extend(is_correct)
    
    # Validation
    for x, y in val_dl:
        val_epoch_losses.append(val_loss(x, y, model, loss_fn))
        is_correct = accuracy(x, y, model)
        val_epoch_accuracies.extend(is_correct)
    
    # Tính trung bình
    train_losses.append(np.mean(train_epoch_losses))
    train_accuracies.append(np.mean(train_epoch_accuracies))
    val_losses.append(np.mean(val_epoch_losses))
    val_accuracies.append(np.mean(val_epoch_accuracies))

# 9. Vẽ kết quả và lưu file
epochs = np.arange(5)+1
plt.figure(figsize=(12,8))

plt.subplot(211)
plt.plot(epochs, train_losses, 'bo-', label='Training loss')
plt.plot(epochs, val_losses, 'go-', label='Validation loss')
plt.title('Training and validation loss (batch size=32)')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()

plt.subplot(212)
plt.plot(epochs, train_accuracies, 'bo-', label='Training accuracy')
plt.plot(epochs, val_accuracies, 'go-', label='Validation accuracy')
plt.title('Training and validation accuracy (batch size=32)')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.gca().set_yticklabels(['{:.0f}%'.format(x*100) for x in plt.gca().get_yticks()])
plt.legend()

plt.tight_layout()
plt.savefig("train_val_results.png")
print("✅ Đã lưu biểu đồ huấn luyện/kiểm định vào train_val_results.png")


Train: torch.Size([60000, 28, 28]) torch.Size([60000])
Validation: torch.Size([10000, 28, 28]) torch.Size([10000])
Classes: ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat', 'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Epoch: 1
Epoch: 2
Epoch: 3
Epoch: 4
Epoch: 5
✅ Đã lưu biểu đồ huấn luyện/kiểm định vào train_val_results.png


C:\Users\Admin\AppData\Local\Temp\ipykernel_22520\2450082495.py:133: UserWarning: FixedFormatter should only be used together with FixedLocator
  plt.gca().set_yticklabels(['{:.0f}%'.format(x*100) for x in plt.gca().get_yticks()])


In [14]:
# Cài đặt thư viện cần thiết
!pip install --upgrade torch torchvision matplotlib matplotlib-inline numpy

# Import
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
import matplotlib
matplotlib.use("Agg")  # tránh lỗi backend GUI
import matplotlib.pyplot as plt
import numpy as np
from torchvision import datasets

# Thiết bị
device = "cuda" if torch.cuda.is_available() else "cpu"

# 1. Tải dữ liệu FashionMNIST (train + validation)
data_folder = '~/data/FMNIST'
fmnist = datasets.FashionMNIST(data_folder, download=True, train=True)
val_fmnist = datasets.FashionMNIST(data_folder, download=True, train=False)

tr_images, tr_targets = fmnist.data, fmnist.targets
val_images, val_targets = val_fmnist.data, val_fmnist.targets

print("Train:", tr_images.shape, tr_targets.shape)
print("Validation:", val_images.shape, val_targets.shape)
print("Classes:", fmnist.classes)

# 2. Dataset tùy chỉnh
class FMNISTDataset(Dataset):
    def __init__(self, x, y):
        x = x.float()/255.0
        x = x.view(-1, 28*28)
        self.x, self.y = x, y
    def __getitem__(self, ix):
        return self.x[ix].to(device), self.y[ix].to(device)
    def __len__(self):
        return len(self.x)

# 3. DataLoader
def get_data():
    train = FMNISTDataset(tr_images, tr_targets)
    trn_dl = DataLoader(train, batch_size=32, shuffle=True)
    val = FMNISTDataset(val_images, val_targets)
    val_dl = DataLoader(val, batch_size=len(val_images), shuffle=False)
    return trn_dl, val_dl

# 4. Mô hình, loss, optimizer
def get_model():
    model = nn.Sequential(
        nn.Linear(28*28, 1000),
        nn.ReLU(),
        nn.Linear(1000, 10)
    ).to(device)
    loss_fn = nn.CrossEntropyLoss()
    optimizer = Adam(model.parameters(), lr=1e-2)
    return model, loss_fn, optimizer

# 5. Train batch
def train_batch(x, y, model, opt, loss_fn):
    model.train()
    prediction = model(x)
    batch_loss = loss_fn(prediction, y)
    batch_loss.backward()
    opt.step()
    opt.zero_grad()
    return batch_loss.item()

# 6. Accuracy
@torch.no_grad()
def accuracy(x, y, model):
    model.eval()
    prediction = model(x)
    _, argmaxes = prediction.max(-1)
    return (argmaxes == y).cpu().numpy().tolist()

# 7. Validation loss
@torch.no_grad()
def val_loss(X, y, model, loss_fn):
    prediction = model(X)
    return loss_fn(prediction, y).item()

# 8. Huấn luyện + đánh giá
trn_dl, val_dl = get_data()
model, loss_fn, optimizer = get_model()

train_losses, train_accuracies = [], []
val_losses, val_accuracies = [], []

for epoch in range(5):
    print("Epoch:", epoch+1)
    train_epoch_losses, train_epoch_accuracies = [], []
    val_epoch_losses, val_epoch_accuracies = [], []
    
    # Train
    for x, y in trn_dl:
        batch_loss = train_batch(x, y, model, optimizer, loss_fn)
        train_epoch_losses.append(batch_loss)
        is_correct = accuracy(x, y, model)
        train_epoch_accuracies.extend(is_correct)
    
    # Validation
    for x, y in val_dl:
        val_epoch_losses.append(val_loss(x, y, model, loss_fn))
        is_correct = accuracy(x, y, model)
        val_epoch_accuracies.extend(is_correct)
    
    # Tính trung bình
    train_losses.append(np.mean(train_epoch_losses))
    train_accuracies.append(np.mean(train_epoch_accuracies))
    val_losses.append(np.mean(val_epoch_losses))
    val_accuracies.append(np.mean(val_epoch_accuracies))

# 9. Vẽ kết quả và lưu file
epochs = np.arange(5)+1
plt.figure(figsize=(12,8))

plt.subplot(211)
plt.plot(epochs, train_losses, 'bo-', label='Training loss')
plt.plot(epochs, val_losses, 'go-', label='Validation loss')
plt.title('Training and validation loss (batch size=32)')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()

plt.subplot(212)
plt.plot(epochs, train_accuracies, 'bo-', label='Training accuracy')
plt.plot(epochs, val_accuracies, 'go-', label='Validation accuracy')
plt.title('Training and validation accuracy (batch size=32)')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.gca().set_yticklabels(['{:.0f}%'.format(x*100) for x in plt.gca().get_yticks()])
plt.legend()

plt.tight_layout()
plt.savefig("train_val_results.png")
print("✅ Đã lưu biểu đồ huấn luyện/kiểm định vào train_val_results.png")


Train: torch.Size([60000, 28, 28]) torch.Size([60000])
Validation: torch.Size([10000, 28, 28]) torch.Size([10000])
Classes: ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat', 'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']
Epoch: 1



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Epoch: 2
Epoch: 3
Epoch: 4
Epoch: 5
✅ Đã lưu biểu đồ huấn luyện/kiểm định vào train_val_results.png


C:\Users\Admin\AppData\Local\Temp\ipykernel_22520\2450082495.py:133: UserWarning: FixedFormatter should only be used together with FixedLocator
  plt.gca().set_yticklabels(['{:.0f}%'.format(x*100) for x in plt.gca().get_yticks()])


In [15]:
# Cài đặt thư viện cần thiết
!pip install --upgrade torch torchvision matplotlib matplotlib-inline numpy

# Import
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import SGD
import matplotlib
matplotlib.use("Agg")  # tránh lỗi backend GUI
import matplotlib.pyplot as plt
import numpy as np
from torchvision import datasets

# Thiết bị
device = "cuda" if torch.cuda.is_available() else "cpu"

# 1. Tải dữ liệu FashionMNIST (train + validation)
data_folder = '~/data/FMNIST'
fmnist = datasets.FashionMNIST(data_folder, download=True, train=True)
val_fmnist = datasets.FashionMNIST(data_folder, download=True, train=False)

tr_images, tr_targets = fmnist.data, fmnist.targets
val_images, val_targets = val_fmnist.data, val_fmnist.targets

print("Train:", tr_images.shape, tr_targets.shape)
print("Validation:", val_images.shape, val_targets.shape)
print("Classes:", fmnist.classes)

# 2. Dataset tùy chỉnh
class FMNISTDataset(Dataset):
    def __init__(self, x, y):
        x = x.float()/255.0
        x = x.view(-1, 28*28)
        self.x, self.y = x, y
    def __getitem__(self, ix):
        return self.x[ix].to(device), self.y[ix].to(device)
    def __len__(self):
        return len(self.x)

# 3. DataLoader
def get_data():
    train = FMNISTDataset(tr_images, tr_targets)
    trn_dl = DataLoader(train, batch_size=32, shuffle=True)
    val = FMNISTDataset(val_images, val_targets)
    val_dl = DataLoader(val, batch_size=len(val_images), shuffle=False)
    return trn_dl, val_dl

# 4. Mô hình, loss, optimizer
def get_model():
    model = nn.Sequential(
        nn.Linear(28*28, 1000),
        nn.ReLU(),
        nn.Linear(1000, 10)
    ).to(device)
    loss_fn = nn.CrossEntropyLoss()
    optimizer = SGD(model.parameters(), lr=1e-2)
    return model, loss_fn, optimizer

# 5. Train batch
def train_batch(x, y, model, opt, loss_fn):
    model.train()
    prediction = model(x)
    batch_loss = loss_fn(prediction, y)
    batch_loss.backward()
    opt.step()
    opt.zero_grad()
    return batch_loss.item()

# 6. Accuracy
@torch.no_grad()
def accuracy(x, y, model):
    model.eval()
    prediction = model(x)
    _, argmaxes = prediction.max(-1)
    return (argmaxes == y).cpu().numpy().tolist()

# 7. Validation loss
@torch.no_grad()
def val_loss(X, y, model, loss_fn):
    prediction = model(X)
    return loss_fn(prediction, y).item()

# 8. Huấn luyện + đánh giá
trn_dl, val_dl = get_data()
model, loss_fn, optimizer = get_model()

train_losses, train_accuracies = [], []
val_losses, val_accuracies = [], []

for epoch in range(5):
    print("Epoch:", epoch+1)
    train_epoch_losses, train_epoch_accuracies = [], []
    val_epoch_losses, val_epoch_accuracies = [], []
    
    # Train
    for x, y in trn_dl:
        batch_loss = train_batch(x, y, model, optimizer, loss_fn)
        train_epoch_losses.append(batch_loss)
        is_correct = accuracy(x, y, model)
        train_epoch_accuracies.extend(is_correct)
    
    # Validation
    for x, y in val_dl:
        val_epoch_losses.append(val_loss(x, y, model, loss_fn))
        is_correct = accuracy(x, y, model)
        val_epoch_accuracies.extend(is_correct)
    
    # Tính trung bình
    train_losses.append(np.mean(train_epoch_losses))
    train_accuracies.append(np.mean(train_epoch_accuracies))
    val_losses.append(np.mean(val_epoch_losses))
    val_accuracies.append(np.mean(val_epoch_accuracies))

# 9. Vẽ kết quả và lưu file
epochs = np.arange(5)+1
plt.figure(figsize=(12,8))

plt.subplot(211)
plt.plot(epochs, train_losses, 'bo-', label='Training loss')
plt.plot(epochs, val_losses, 'r^-', label='Validation loss')
plt.title('Training and validation loss (batch size=32)')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()

plt.subplot(212)
plt.plot(epochs, train_accuracies, 'bo-', label='Training accuracy')
plt.plot(epochs, val_accuracies, 'r^-', label='Validation accuracy')
plt.title('Training and validation accuracy (batch size=32)')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.gca().set_yticklabels(['{:.0f}%'.format(x*100) for x in plt.gca().get_yticks()])
plt.legend()

plt.tight_layout()
plt.savefig("train_val_results.png")
print("✅ Đã lưu biểu đồ huấn luyện/kiểm định vào train_val_results.png")


Train: torch.Size([60000, 28, 28]) torch.Size([60000])
Validation: torch.Size([10000, 28, 28]) torch.Size([10000])
Classes: ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat', 'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']
Epoch: 1



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Epoch: 2
Epoch: 3
Epoch: 4
Epoch: 5
✅ Đã lưu biểu đồ huấn luyện/kiểm định vào train_val_results.png


C:\Users\Admin\AppData\Local\Temp\ipykernel_22520\3866946038.py:133: UserWarning: FixedFormatter should only be used together with FixedLocator
  plt.gca().set_yticklabels(['{:.0f}%'.format(x*100) for x in plt.gca().get_yticks()])


In [16]:
# Cài đặt thư viện cần thiết
!pip install --upgrade torch torchvision matplotlib matplotlib-inline numpy

# Import
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
import matplotlib
matplotlib.use("Agg")  # tránh lỗi backend GUI
import matplotlib.pyplot as plt
import numpy as np
from torchvision import datasets

# Thiết bị
device = "cuda" if torch.cuda.is_available() else "cpu"

# 1. Tải dữ liệu FashionMNIST (train + validation)
data_folder = '~/data/FMNIST'
fmnist = datasets.FashionMNIST(data_folder, download=True, train=True)
val_fmnist = datasets.FashionMNIST(data_folder, download=True, train=False)

tr_images, tr_targets = fmnist.data, fmnist.targets
val_images, val_targets = val_fmnist.data, val_fmnist.targets

print("Train:", tr_images.shape, tr_targets.shape)
print("Validation:", val_images.shape, val_targets.shape)
print("Classes:", fmnist.classes)

# 2. Dataset tùy chỉnh
class FMNISTDataset(Dataset):
    def __init__(self, x, y):
        x = x.float()/255.0
        x = x.view(-1, 28*28)
        self.x, self.y = x, y
    def __getitem__(self, ix):
        return self.x[ix].to(device), self.y[ix].to(device)
    def __len__(self):
        return len(self.x)

# 3. DataLoader
def get_data():
    train = FMNISTDataset(tr_images, tr_targets)
    trn_dl = DataLoader(train, batch_size=32, shuffle=True)
    val = FMNISTDataset(val_images, val_targets)
    val_dl = DataLoader(val, batch_size=len(val_images), shuffle=False)
    return trn_dl, val_dl

# 4. Mô hình, loss, optimizer
def get_model():
    model = nn.Sequential(
        nn.Linear(28*28, 1000),
        nn.ReLU(),
        nn.Linear(1000, 10)
    ).to(device)
    loss_fn = nn.CrossEntropyLoss()
    optimizer = Adam(model.parameters(), lr=1e-2)
    return model, loss_fn, optimizer

# 5. Train batch
def train_batch(x, y, model, opt, loss_fn):
    model.train()
    prediction = model(x)
    batch_loss = loss_fn(prediction, y)
    batch_loss.backward()
    opt.step()
    opt.zero_grad()
    return batch_loss.item()

# 6. Accuracy
@torch.no_grad()
def accuracy(x, y, model):
    model.eval()
    prediction = model(x)
    _, argmaxes = prediction.max(-1)
    return (argmaxes == y).cpu().numpy().tolist()

# 7. Validation loss
@torch.no_grad()
def val_loss(X, y, model, loss_fn):
    prediction = model(X)
    return loss_fn(prediction, y).item()

# 8. Huấn luyện + đánh giá
trn_dl, val_dl = get_data()
model, loss_fn, optimizer = get_model()

train_losses, train_accuracies = [], []
val_losses, val_accuracies = [], []

for epoch in range(5):
    print("Epoch:", epoch+1)
    train_epoch_losses, train_epoch_accuracies = [], []
    val_epoch_losses, val_epoch_accuracies = [], []
    
    # Train
    for x, y in trn_dl:
        batch_loss = train_batch(x, y, model, optimizer, loss_fn)
        train_epoch_losses.append(batch_loss)
        is_correct = accuracy(x, y, model)
        train_epoch_accuracies.extend(is_correct)
    
    # Validation
    for x, y in val_dl:
        val_epoch_losses.append(val_loss(x, y, model, loss_fn))
        is_correct = accuracy(x, y, model)
        val_epoch_accuracies.extend(is_correct)
    
    # Tính trung bình
    train_losses.append(np.mean(train_epoch_losses))
    train_accuracies.append(np.mean(train_epoch_accuracies))
    val_losses.append(np.mean(val_epoch_losses))
    val_accuracies.append(np.mean(val_epoch_accuracies))

# 9. Vẽ kết quả và lưu file
epochs = np.arange(5)+1
plt.figure(figsize=(12,8))

plt.subplot(211)
plt.plot(epochs, train_losses, 'bo-', label='Training loss')
plt.plot(epochs, val_losses, 'r^-', label='Validation loss')
plt.title('Training and validation loss (batch size=32)')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()

plt.subplot(212)
plt.plot(epochs, train_accuracies, 'bo-', label='Training accuracy')
plt.plot(epochs, val_accuracies, 'r^-', label='Validation accuracy')
plt.title('Training and validation accuracy (batch size=32)')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.gca().set_yticklabels(['{:.0f}%'.format(x*100) for x in plt.gca().get_yticks()])
plt.legend()

plt.tight_layout()
plt.savefig("train_val_results.png")
print("✅ Đã lưu biểu đồ huấn luyện/kiểm định vào train_val_results.png")


Train: torch.Size([60000, 28, 28]) torch.Size([60000])
Validation: torch.Size([10000, 28, 28]) torch.Size([10000])
Classes: ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat', 'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']
Epoch: 1



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Epoch: 2
Epoch: 3
Epoch: 4
Epoch: 5
✅ Đã lưu biểu đồ huấn luyện/kiểm định vào train_val_results.png


C:\Users\Admin\AppData\Local\Temp\ipykernel_22520\1899208889.py:133: UserWarning: FixedFormatter should only be used together with FixedLocator
  plt.gca().set_yticklabels(['{:.0f}%'.format(x*100) for x in plt.gca().get_yticks()])


In [17]:
# Cài đặt thư viện cần thiết
!pip install --upgrade torch torchvision matplotlib matplotlib-inline numpy

# Import
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
import matplotlib
matplotlib.use("Agg")  # tránh lỗi backend GUI
import matplotlib.pyplot as plt
import numpy as np
from torchvision import datasets

# Thiết bị
device = "cuda" if torch.cuda.is_available() else "cpu"

# 1. Tải dữ liệu FashionMNIST (train + validation)
data_folder = './data/FMNIST'
fmnist = datasets.FashionMNIST(data_folder, download=True, train=True)
val_fmnist = datasets.FashionMNIST(data_folder, download=True, train=False)

tr_images, tr_targets = fmnist.data, fmnist.targets
val_images, val_targets = val_fmnist.data, val_fmnist.targets

print("Train:", tr_images.shape, tr_targets.shape)
print("Validation:", val_images.shape, val_targets.shape)
print("Classes:", fmnist.classes)

# 2. Dataset tùy chỉnh
class FMNISTDataset(Dataset):
    def __init__(self, x, y):
        x = x.float()/255.0
        x = x.view(-1, 28*28)
        self.x, self.y = x, y
    def __getitem__(self, ix):
        return self.x[ix].to(device), self.y[ix].to(device)
    def __len__(self):
        return len(self.x)

# 3. DataLoader
def get_data():
    train = FMNISTDataset(tr_images, tr_targets)
    trn_dl = DataLoader(train, batch_size=32, shuffle=True)
    val = FMNISTDataset(val_images, val_targets)
    val_dl = DataLoader(val, batch_size=len(val_images), shuffle=False)
    return trn_dl, val_dl

# 4. Mô hình, loss, optimizer
def get_model():
    model = nn.Sequential(
        nn.Linear(28*28, 1000),
        nn.ReLU(),
        nn.Linear(1000, 10)
    ).to(device)
    loss_fn = nn.CrossEntropyLoss()
    optimizer = Adam(model.parameters(), lr=1e-2)
    return model, loss_fn, optimizer

# 5. Train batch
def train_batch(x, y, model, opt, loss_fn):
    model.train()
    prediction = model(x)
    batch_loss = loss_fn(prediction, y)
    batch_loss.backward()
    opt.step()
    opt.zero_grad()
    return batch_loss.item()

# 6. Accuracy
@torch.no_grad()
def accuracy(x, y, model):
    model.eval()
    prediction = model(x)
    _, argmaxes = prediction.max(-1)
    return (argmaxes == y).cpu().numpy().tolist()

# 7. Validation loss
@torch.no_grad()
def val_loss(X, y, model, loss_fn):
    prediction = model(X)
    return loss_fn(prediction, y).item()

# 8. Huấn luyện + đánh giá
trn_dl, val_dl = get_data()
model, loss_fn, optimizer = get_model()

train_losses, train_accuracies = [], []
val_losses, val_accuracies = [], []

for epoch in range(5):
    print("Epoch:", epoch+1)
    train_epoch_losses, train_epoch_accuracies = [], []
    val_epoch_losses, val_epoch_accuracies = [], []
    
    # Train
    for x, y in trn_dl:
        batch_loss = train_batch(x, y, model, optimizer, loss_fn)
        train_epoch_losses.append(batch_loss)
        is_correct = accuracy(x, y, model)
        train_epoch_accuracies.extend(is_correct)
    
    # Validation
    for x, y in val_dl:
        val_epoch_losses.append(val_loss(x, y, model, loss_fn))
        is_correct = accuracy(x, y, model)
        val_epoch_accuracies.extend(is_correct)
    
    # Tính trung bình
    train_losses.append(np.mean(train_epoch_losses))
    train_accuracies.append(np.mean(train_epoch_accuracies))
    val_losses.append(np.mean(val_epoch_losses))
    val_accuracies.append(np.mean(val_epoch_accuracies))

# 9. Vẽ kết quả và lưu file
epochs = np.arange(5)+1
plt.figure(figsize=(12,8))

plt.subplot(121)
plt.plot(epochs, train_losses, 'bo-', label='Training loss')
plt.plot(epochs, val_losses, 'r^-', label='Validation loss')
plt.title('Training and validation loss (batch size=32)')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()

plt.subplot(122)
plt.plot(epochs, train_accuracies, 'bo-', label='Training accuracy')
plt.plot(epochs, val_accuracies, 'r^-', label='Validation accuracy')
plt.title('Training and validation accuracy (batch size=32)')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.gca().set_yticklabels(['{:.0f}%'.format(x*100) for x in plt.gca().get_yticks()])
plt.legend()

plt.tight_layout()
plt.savefig("train_val_results.png")
print("✅ Đã lưu biểu đồ huấn luyện/kiểm định vào train_val_results.png")



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


100%|██████████| 26.4M/26.4M [00:16<00:00, 1.60MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 149kB/s]
100%|██████████| 4.42M/4.42M [00:02<00:00, 2.03MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 4.55MB/s]


Train: torch.Size([60000, 28, 28]) torch.Size([60000])
Validation: torch.Size([10000, 28, 28]) torch.Size([10000])
Classes: ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat', 'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']
Epoch: 1
Epoch: 2
Epoch: 3
Epoch: 4
Epoch: 5
✅ Đã lưu biểu đồ huấn luyện/kiểm định vào train_val_results.png


C:\Users\Admin\AppData\Local\Temp\ipykernel_22520\3121375386.py:133: UserWarning: FixedFormatter should only be used together with FixedLocator
  plt.gca().set_yticklabels(['{:.0f}%'.format(x*100) for x in plt.gca().get_yticks()])


In [18]:
# Cài đặt thư viện cần thiết
!pip install --upgrade torch torchvision matplotlib matplotlib-inline numpy

# Import
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
import matplotlib
matplotlib.use("Agg")  # tránh lỗi backend GUI
import matplotlib.pyplot as plt
import numpy as np
from torchvision import datasets

# Thiết bị
device = "cuda" if torch.cuda.is_available() else "cpu"

# 1. Tải dữ liệu FashionMNIST (train + validation)
data_folder = './data/FMNIST'
fmnist = datasets.FashionMNIST(data_folder, download=True, train=True)
val_fmnist = datasets.FashionMNIST(data_folder, download=True, train=False)

tr_images, tr_targets = fmnist.data, fmnist.targets
val_images, val_targets = val_fmnist.data, val_fmnist.targets

print("Train:", tr_images.shape, tr_targets.shape)
print("Validation:", val_images.shape, val_targets.shape)
print("Classes:", fmnist.classes)

# 2. Dataset tùy chỉnh
class FMNISTDataset(Dataset):
    def __init__(self, x, y):
        x = x.float()/255.0
        x = x.view(-1, 28*28)
        self.x, self.y = x, y
    def __getitem__(self, ix):
        return self.x[ix].to(device), self.y[ix].to(device)
    def __len__(self):
        return len(self.x)

# 3. DataLoader
def get_data():
    train = FMNISTDataset(tr_images, tr_targets)
    trn_dl = DataLoader(train, batch_size=32, shuffle=True)
    val = FMNISTDataset(val_images, val_targets)
    val_dl = DataLoader(val, batch_size=len(val_images), shuffle=False)
    return trn_dl, val_dl

# 4. Mô hình, loss, optimizer
def get_model():
    model = nn.Sequential(
        nn.Linear(28*28, 1000),
        nn.ReLU(),
        nn.Linear(1000, 10)
    ).to(device)
    loss_fn = nn.CrossEntropyLoss()
    optimizer = Adam(model.parameters(), lr=1e-2)
    return model, loss_fn, optimizer

# 5. Train batch
def train_batch(x, y, model, opt, loss_fn):
    model.train()
    prediction = model(x)
    batch_loss = loss_fn(prediction, y)
    batch_loss.backward()
    opt.step()
    opt.zero_grad()
    return batch_loss.item()

# 6. Accuracy
@torch.no_grad()
def accuracy(x, y, model):
    model.eval()
    prediction = model(x)
    _, argmaxes = prediction.max(-1)
    return (argmaxes == y).cpu().numpy().tolist()

# 7. Validation loss
@torch.no_grad()
def val_loss(X, y, model, loss_fn):
    prediction = model(X)
    return loss_fn(prediction, y).item()

# 8. Huấn luyện + đánh giá
trn_dl, val_dl = get_data()
model, loss_fn, optimizer = get_model()

train_losses, train_accuracies = [], []
val_losses, val_accuracies = [], []

for epoch in range(5):
    print("Epoch:", epoch+1)
    train_epoch_losses, train_epoch_accuracies = [], []
    val_epoch_losses, val_epoch_accuracies = [], []
    
    # Train
    for x, y in trn_dl:
        batch_loss = train_batch(x, y, model, optimizer, loss_fn)
        train_epoch_losses.append(batch_loss)
        is_correct = accuracy(x, y, model)
        train_epoch_accuracies.extend(is_correct)
    
    # Validation
    for x, y in val_dl:
        val_epoch_losses.append(val_loss(x, y, model, loss_fn))
        is_correct = accuracy(x, y, model)
        val_epoch_accuracies.extend(is_correct)
    
    # Tính trung bình
    train_losses.append(np.mean(train_epoch_losses))
    train_accuracies.append(np.mean(train_epoch_accuracies))
    val_losses.append(np.mean(val_epoch_losses))
    val_accuracies.append(np.mean(val_epoch_accuracies))

# 9. Vẽ kết quả và lưu file
epochs = np.arange(5)+1
plt.figure(figsize=(12,8))

plt.subplot(121)
plt.plot(epochs, train_losses, 'bo-', label='Training loss')
plt.plot(epochs, val_losses, 'r^-', label='Validation loss')
plt.title('Training and validation loss (batch size=32)')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()

plt.subplot(122)
plt.plot(epochs, train_accuracies, 'bo-', label='Training accuracy')
plt.plot(epochs, val_accuracies, 'r^-', label='Validation accuracy')
plt.title('Training and validation accuracy (batch size=32)')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.gca().set_yticklabels(['{:.0f}%'.format(x*100) for x in plt.gca().get_yticks()])
plt.legend()

plt.tight_layout()
plt.savefig("train_val_results.png")
print("✅ Đã lưu biểu đồ huấn luyện/kiểm định vào train_val_results.png")



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Train: torch.Size([60000, 28, 28]) torch.Size([60000])
Validation: torch.Size([10000, 28, 28]) torch.Size([10000])
Classes: ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat', 'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']
Epoch: 1
Epoch: 2
Epoch: 3
Epoch: 4
Epoch: 5
✅ Đã lưu biểu đồ huấn luyện/kiểm định vào train_val_results.png


C:\Users\Admin\AppData\Local\Temp\ipykernel_22520\3121375386.py:133: UserWarning: FixedFormatter should only be used together with FixedLocator
  plt.gca().set_yticklabels(['{:.0f}%'.format(x*100) for x in plt.gca().get_yticks()])


In [19]:
# Cài đặt thư viện cần thiết
!pip install --upgrade torch torchvision matplotlib matplotlib-inline numpy

# Import
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
import matplotlib
matplotlib.use("Agg")  # tránh lỗi backend GUI
import matplotlib.pyplot as plt
import numpy as np
from torchvision import datasets

# Thiết bị
device = "cuda" if torch.cuda.is_available() else "cpu"

# 1. Tải dữ liệu FashionMNIST (train + validation)
data_folder = './data/FMNIST'
train_fmnist = datasets.FashionMNIST(data_folder, download=True, train=True)
val_fmnist = datasets.FashionMNIST(data_folder, download=True, train=False)

tr_images, tr_targets = train_fmnist.data, train_fmnist.targets
val_images, val_targets = val_fmnist.data, val_fmnist.targets

print("Train:", tr_images.shape, tr_targets.shape)
print("Validation:", val_images.shape, val_targets.shape)
print("Classes:", train_fmnist.classes)

# 2. Dataset tùy chỉnh
class FMNISTDataset(Dataset):
    def __init__(self, x, y):
        x = x.float()/255.0
        x = x.view(-1, 28*28)
        self.x, self.y = x, y
    def __getitem__(self, ix):
        return self.x[ix].to(device), self.y[ix].to(device)
    def __len__(self):
        return len(self.x)

# 3. DataLoader
def get_data():
    train = FMNISTDataset(tr_images, tr_targets)
    trn_dl = DataLoader(train, batch_size=32, shuffle=True)
    val = FMNISTDataset(val_images, val_targets)
    val_dl = DataLoader(val, batch_size=len(val_images), shuffle=False)
    return trn_dl, val_dl

# 4. Mô hình, loss, optimizer
def get_model():
    model = nn.Sequential(
        nn.Linear(28*28, 1000),
        nn.ReLU(),
        nn.Linear(1000, 10)
    ).to(device)
    loss_fn = nn.CrossEntropyLoss()
    optimizer = Adam(model.parameters(), lr=1e-2)
    return model, loss_fn, optimizer

# 5. Train batch
def train_batch(x, y, model, opt, loss_fn):
    model.train()
    prediction = model(x)
    batch_loss = loss_fn(prediction, y)
    batch_loss.backward()
    opt.step()
    opt.zero_grad()
    return batch_loss.item()

# 6. Accuracy
@torch.no_grad()
def accuracy(x, y, model):
    model.eval()
    prediction = model(x)
    _, argmaxes = prediction.max(-1)
    return (argmaxes == y).cpu().numpy().tolist()

# 7. Validation loss
@torch.no_grad()
def val_loss(X, y, model, loss_fn):
    prediction = model(X)
    return loss_fn(prediction, y).item()

# 8. Huấn luyện + đánh giá
trn_dl, val_dl = get_data()
model, loss_fn, optimizer = get_model()

train_losses, train_accuracies = [], []
val_losses, val_accuracies = [], []

for epoch in range(5):
    print("Epoch:", epoch+1)
    train_epoch_losses, train_epoch_accuracies = [], []
    val_epoch_losses, val_epoch_accuracies = [], []
    
    # Train
    for x, y in trn_dl:
        batch_loss = train_batch(x, y, model, optimizer, loss_fn)
        train_epoch_losses.append(batch_loss)
        is_correct = accuracy(x, y, model)
        train_epoch_accuracies.extend(is_correct)
    
    # Validation
    for x, y in val_dl:
        val_epoch_losses.append(val_loss(x, y, model, loss_fn))
        is_correct = accuracy(x, y, model)
        val_epoch_accuracies.extend(is_correct)
    
    # Tính trung bình
    train_losses.append(np.mean(train_epoch_losses))
    train_accuracies.append(np.mean(train_epoch_accuracies))
    val_losses.append(np.mean(val_epoch_losses))
    val_accuracies.append(np.mean(val_epoch_accuracies))

# 9. Vẽ kết quả và lưu file
epochs = np.arange(5)+1
plt.figure(figsize=(12,8))

plt.subplot(121)
plt.plot(epochs, train_losses, 'bo-', label='Training loss')
plt.plot(epochs, val_losses, 'r^-', label='Validation loss')
plt.title('Training and validation loss (batch size=32)')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()

plt.subplot(122)
plt.plot(epochs, train_accuracies, 'bo-', label='Training accuracy')
plt.plot(epochs, val_accuracies, 'r^-', label='Validation accuracy')
plt.title('Training and validation accuracy (batch size=32)')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.gca().set_yticklabels(['{:.0f}%'.format(x*100) for x in plt.gca().get_yticks()])
plt.legend()

plt.tight_layout()
plt.savefig("train_val_results.png")
print("✅ Đã lưu biểu đồ huấn luyện/kiểm định vào train_val_results.png")


Train: torch.Size([60000, 28, 28]) torch.Size([60000])
Validation: torch.Size([10000, 28, 28]) torch.Size([10000])
Classes: ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat', 'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']
Epoch: 1



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Epoch: 2
Epoch: 3
Epoch: 4
Epoch: 5
✅ Đã lưu biểu đồ huấn luyện/kiểm định vào train_val_results.png


C:\Users\Admin\AppData\Local\Temp\ipykernel_22520\2293659254.py:133: UserWarning: FixedFormatter should only be used together with FixedLocator
  plt.gca().set_yticklabels(['{:.0f}%'.format(x*100) for x in plt.gca().get_yticks()])


In [20]:
# Cài đặt thư viện cần thiết
!pip install --upgrade torch torchvision matplotlib matplotlib-inline numpy

# Import
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
import matplotlib
matplotlib.use("Agg")  # tránh lỗi backend GUI
import matplotlib.pyplot as plt
import numpy as np
from torchvision import datasets

# Thiết bị
device = "cuda" if torch.cuda.is_available() else "cpu"

# 1. Tải dữ liệu FashionMNIST (train + validation)
data_folder = './data/FMNIST'
train_fmnist = datasets.FashionMNIST(data_folder, download=True, train=True)
val_fmnist = datasets.FashionMNIST(data_folder, download=True, train=False)

tr_images, tr_targets = train_fmnist.data, train_fmnist.targets
val_images, val_targets = val_fmnist.data, val_fmnist.targets

print("Train:", tr_images.shape, tr_targets.shape)
print("Validation:", val_images.shape, val_targets.shape)
print("Classes:", train_fmnist.classes)

# 2. Dataset tùy chỉnh
class FMNISTDataset(Dataset):
    def __init__(self, x, y):
        x = x.float()/255.0
        x = x.view(-1, 28*28)
        self.x, self.y = x, y
    def __getitem__(self, ix):
        return self.x[ix].to(device), self.y[ix].to(device)
    def __len__(self):
        return len(self.x)

# 3. DataLoader
def get_data():
    train = FMNISTDataset(tr_images, tr_targets)
    trn_dl = DataLoader(train, batch_size=32, shuffle=True)
    val = FMNISTDataset(val_images, val_targets)
    val_dl = DataLoader(val, batch_size=len(val_images), shuffle=False)
    return trn_dl, val_dl

# 4. Mô hình, loss, optimizer
def get_model():
    model = nn.Sequential(
        nn.Linear(28*28, 1000),
        nn.ReLU(),
        nn.Linear(1000, 10)
    ).to(device)
    loss_fn = nn.CrossEntropyLoss()
    optimizer = Adam(model.parameters(), lr=1e-2)
    return model, loss_fn, optimizer

# 5. Train batch
def train_batch(x, y, model, opt, loss_fn):
    model.train()
    prediction = model(x)
    batch_loss = loss_fn(prediction, y)
    batch_loss.backward()
    opt.step()
    opt.zero_grad()
    return batch_loss.item()

# 6. Accuracy
@torch.no_grad()
def accuracy(x, y, model):
    model.eval()
    prediction = model(x)
    _, argmaxes = prediction.max(-1)
    return (argmaxes == y).cpu().numpy().tolist()

# 7. Validation loss
@torch.no_grad()
def val_loss(X, y, model, loss_fn):
    prediction = model(X)
    return loss_fn(prediction, y).item()

# 8. Huấn luyện + đánh giá
trn_dl, val_dl = get_data()
model, loss_fn, optimizer = get_model()

train_losses, train_accuracies = [], []
val_losses, val_accuracies = [], []

for epoch in range(5):
    print("Epoch:", epoch+1)
    train_epoch_losses, train_epoch_accuracies = [], []
    val_epoch_losses, val_epoch_accuracies = [], []
    
    # Train
    for x, y in trn_dl:
        batch_loss = train_batch(x, y, model, optimizer, loss_fn)
        train_epoch_losses.append(batch_loss)
        is_correct = accuracy(x, y, model)
        train_epoch_accuracies.extend(is_correct)
    
    # Validation
    for x, y in val_dl:
        val_epoch_losses.append(val_loss(x, y, model, loss_fn))
        is_correct = accuracy(x, y, model)
        val_epoch_accuracies.extend(is_correct)
    
    # Tính trung bình
    train_losses.append(np.mean(train_epoch_losses))
    train_accuracies.append(np.mean(train_epoch_accuracies))
    val_losses.append(np.mean(val_epoch_losses))
    val_accuracies.append(np.mean(val_epoch_accuracies))

# 9. Vẽ kết quả và lưu file
epochs = np.arange(5)+1
plt.figure(figsize=(12,8))

plt.subplot(121)
plt.plot(epochs, train_losses, 'bo-', label='Training loss')
plt.plot(epochs, val_losses, 'r^-', label='Validation loss')
plt.title('Training and validation loss (batch size=32)')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()

plt.subplot(122)
plt.plot(epochs, train_accuracies, 'bo-', label='Training accuracy')
plt.plot(epochs, val_accuracies, 'r^-', label='Validation accuracy')
plt.title('Training and validation accuracy (batch size=32)')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.gca().set_yticklabels(['{:.0f}%'.format(x*100) for x in plt.gca().get_yticks()])
plt.legend()

plt.tight_layout()
plt.savefig("train_val_results.png")
print("✅ Đã lưu biểu đồ huấn luyện/kiểm định vào train_val_results.png")



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Train: torch.Size([60000, 28, 28]) torch.Size([60000])
Validation: torch.Size([10000, 28, 28]) torch.Size([10000])
Classes: ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat', 'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']
Epoch: 1
Epoch: 2
Epoch: 3
Epoch: 4
Epoch: 5
✅ Đã lưu biểu đồ huấn luyện/kiểm định vào train_val_results.png


C:\Users\Admin\AppData\Local\Temp\ipykernel_22520\2293659254.py:133: UserWarning: FixedFormatter should only be used together with FixedLocator
  plt.gca().set_yticklabels(['{:.0f}%'.format(x*100) for x in plt.gca().get_yticks()])


In [21]:
# Cài đặt thư viện cần thiết
!pip install --upgrade torch torchvision matplotlib matplotlib-inline numpy

# Import
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import SGD
import matplotlib
matplotlib.use("Agg")  # tránh lỗi backend GUI
import matplotlib.pyplot as plt
import numpy as np
from torchvision import datasets

# Thiết bị
device = "cuda" if torch.cuda.is_available() else "cpu"

# 1. Tải dữ liệu FashionMNIST (train + validation)
data_folder = './data/FMNIST'
train_fmnist = datasets.FashionMNIST(data_folder, download=True, train=True)
val_fmnist = datasets.FashionMNIST(data_folder, download=True, train=False)

tr_images, tr_targets = train_fmnist.data, train_fmnist.targets
val_images, val_targets = val_fmnist.data, val_fmnist.targets

print("Train:", tr_images.shape, tr_targets.shape)
print("Validation:", val_images.shape, val_targets.shape)
print("Classes:", train_fmnist.classes)

# 2. Dataset tùy chỉnh
class FMNISTDataset(Dataset):
    def __init__(self, x, y):
        x = x.float()/255.0
        x = x.view(-1, 28*28)
        self.x, self.y = x, y
    def __getitem__(self, ix):
        return self.x[ix].to(device), self.y[ix].to(device)
    def __len__(self):
        return len(self.x)

# 3. DataLoader
def get_data():
    train = FMNISTDataset(tr_images, tr_targets)
    trn_dl = DataLoader(train, batch_size=32, shuffle=True)
    val = FMNISTDataset(val_images, val_targets)
    val_dl = DataLoader(val, batch_size=len(val_images), shuffle=False)
    return trn_dl, val_dl

# 4. Mô hình, loss, optimizer
def get_model():
    model = nn.Sequential(
        nn.Linear(28*28, 1000),
        nn.ReLU(),
        nn.Linear(1000, 10)
    ).to(device)
    loss_fn = nn.CrossEntropyLoss()
    optimizer = SGD(model.parameters(), lr=0.01, momentum=0.9)
    return model, loss_fn, optimizer

# 5. Train batch
def train_batch(x, y, model, opt, loss_fn):
    model.train()
    prediction = model(x)
    batch_loss = loss_fn(prediction, y)
    batch_loss.backward()
    opt.step()
    opt.zero_grad()
    return batch_loss.item()

# 6. Accuracy
@torch.no_grad()
def accuracy(x, y, model):
    model.eval()
    prediction = model(x)
    _, argmaxes = prediction.max(-1)
    return (argmaxes == y).cpu().numpy().tolist()

# 7. Validation loss
@torch.no_grad()
def val_loss(X, y, model, loss_fn):
    prediction = model(X)
    return loss_fn(prediction, y).item()

# 8. Huấn luyện + đánh giá
trn_dl, val_dl = get_data()
model, loss_fn, optimizer = get_model()

train_losses, train_accuracies = [], []
val_losses, val_accuracies = [], []

num_epochs = 10
for epoch in range(num_epochs):
    print(f"Epoch {epoch+1}/{num_epochs}")
    train_epoch_losses, train_epoch_accuracies = [], []
    val_epoch_losses, val_epoch_accuracies = [], []
    
    # Train
    for x, y in trn_dl:
        batch_loss = train_batch(x, y, model, optimizer, loss_fn)
        train_epoch_losses.append(batch_loss)
        is_correct = accuracy(x, y, model)
        train_epoch_accuracies.extend(is_correct)
    
    # Validation
    for x, y in val_dl:
        val_epoch_losses.append(val_loss(x, y, model, loss_fn))
        is_correct = accuracy(x, y, model)
        val_epoch_accuracies.extend(is_correct)
    
    # Tính trung bình
    train_losses.append(np.mean(train_epoch_losses))
    train_accuracies.append(np.mean(train_epoch_accuracies))
    val_losses.append(np.mean(val_epoch_losses))
    val_accuracies.append(np.mean(val_epoch_accuracies))

    print(f"Training loss: {train_losses[-1]:.4f}, Training accuracy: {train_accuracies[-1]:.4f}")
    print(f"Validation loss: {val_losses[-1]:.4f}, Validation accuracy: {val_accuracies[-1]:.4f}")

# 9. Vẽ kết quả và lưu file
epochs = np.arange(num_epochs)+1
plt.figure(figsize=(12,8))

plt.subplot(121)
plt.plot(epochs, train_losses, 'bo-', label='Training loss')
plt.plot(epochs, val_losses, 'r^-', label='Validation loss')
plt.title('Training and validation loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()

plt.subplot(122)
plt.plot(epochs, train_accuracies, 'bo-', label='Training accuracy')
plt.plot(epochs, val_accuracies, 'r^-', label='Validation accuracy')
plt.title('Training and validation accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.gca().set_yticklabels(['{:.0f}%'.format(x*100) for x in plt.gca().get_yticks()])
plt.legend()

plt.tight_layout()
plt.savefig("train_val_results.png")
print("✅ Đã lưu biểu đồ huấn luyện/kiểm định vào train_val_results.png")



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Train: torch.Size([60000, 28, 28]) torch.Size([60000])
Validation: torch.Size([10000, 28, 28]) torch.Size([10000])
Classes: ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat', 'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']
Epoch 1/10
Training loss: 0.5352, Training accuracy: 0.8201
Validation loss: 0.4458, Validation accuracy: 0.8425
Epoch 2/10
Training loss: 0.3905, Training accuracy: 0.8676
Validation loss: 0.3972, Validation accuracy: 0.8563
Epoch 3/10
Training loss: 0.3516, Training accuracy: 0.8800
Validation loss: 0.3762, Validation accuracy: 0.8645
Epoch 4/10
Training loss: 0.3269, Training accuracy: 0.8879
Validation loss: 0.3694, Validation accuracy: 0.8666
Epoch 5/10
Training loss: 0.3087, Training accuracy: 0.8937
Validation loss: 0.3475, Validation accuracy: 0.8743
Epoch 6/10
Training loss: 0.2936, Training accuracy: 0.8988
Validation loss: 0.3421, Validation accuracy: 0.8779
Epoch 7/10
Training loss: 0.2821, Training accuracy: 0.9031
Validation loss: 0.3537, Valida

C:\Users\Admin\AppData\Local\Temp\ipykernel_22520\894233161.py:137: UserWarning: FixedFormatter should only be used together with FixedLocator
  plt.gca().set_yticklabels(['{:.0f}%'.format(x*100) for x in plt.gca().get_yticks()])
